# Vision Practice 04. U-Net Segmentation 코드 학습 — 시험 대비 실습본

- 원본: `vision/04_Unet.ipynb`
- 핵심 빈칸 수: **4개**
- `## 정답 입력` 셀을 직접 구현한 뒤 과목별 정답·해설지와 비교하세요.
- API key, 외부 서버 주소, 대용량 데이터 경로는 자신의 환경에 맞게 설정하세요.


# **U-Net — 기본 구조부터 학습 및 시각화까지**

 **U-Net 기반 이미지 세그멘테이션** 실습을 목표로 합니다.  
다운샘플(Encoder) → 업샘플(Decoder) + **Skip Connection** 구조가 **왜 분할(segmentation)에 강한지**를 직접 확인합니다.

## 학습 목표
- U-Net의 **Encoder/Decoder/Skip Connection** 흐름을 코드에서 찾을 수 있다.
- 입력/출력 텐서 shape이 단계별로 어떻게 바뀌는지 설명할 수 있다.
- 간단한 데이터셋으로 **학습 → 검증 → 예측 시각화**까지 한 번에 실행할 수 있다.
> Segmentation은 “어디에 무엇이 있는가”를 픽셀 단위로 예측합니다.  
> 그래서 U-Net은 **공간 정보(spatial detail)** 를 보존/복원하기 위한 구조적 장치를 적극적으로 사용합니다.

---

### 구성 개요

1. U-Net 개요 (이론 정리)
2. 환경 설정 & 데이터 준비
3. PyTorch U-Net 모델 구현
4. 손실함수 / 옵티마이저 설정
5. 학습 루프 (training loop)
6. 평가 및 시각화
7. 과제/확장 아이디어


---
## 0) Setup (환경 준비)

- **목적:** 실습에 필요한 패키지를 로드하고, GPU 사용 여부를 확인합니다.
- **관찰 포인트**
  - `torch.cuda.is_available()` 결과와 사용 디바이스(`cpu/cuda`)
  - (선택) 재현성을 위해 시드를 고정하는 이유


In [ ]:
# 필요한 패키지가 설치되어 있는지 확인하고, 없을 때만 pip로 설치합니다.
# (재실행해도 중복 설치를 피함)
import importlib.util
import sys
import subprocess

def ensure_package(pkg_name: str, import_name=None):
    name = import_name or pkg_name
    if importlib.util.find_spec(name) is None:
        print(f"[install] {pkg_name} (import: {name})")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg_name])
    else:
        print(f"[ok] {pkg_name}")

# U-Net 실습에서 사용: 모델 요약 출력
ensure_package("torchinfo")


import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T

import numpy as np
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# CUDA 상세 정보 (옵션)
if torch.cuda.is_available():
    print("CUDA device count:", torch.cuda.device_count())
    print("Current device index:", torch.cuda.current_device())
    print("Device name:", torch.cuda.get_device_name(0))

---
## 1) 데이터 준비

- **목적:** 실습을 빠르게 진행하기 위해, 간단한 **Synthetic Segmentation Dataset**을 사용합니다.
- **관찰 포인트**
  - `__getitem__`이 반환하는 입력 이미지와 마스크의 shape / dtype
  - 마스크 값이 **0/1**(binary)인지, 혹은 클래스 인덱스 형태인지
  - DataLoader에서 배치로 묶였을 때 텐서 차원이 어떻게 되는지

> 실제 프로젝트에서는 공개 데이터셋(예: 의료 영상, 도로/도시 장면 등)으로 쉽게 교체할 수 있습니다.  
> 여기서는 **모델 구조/학습 흐름**에 집중하기 위해 synthetic 데이터를 사용합니다.


In [ ]:
# ## 정답 입력
# Drill 1: 1) 데이터 준비
# 원본 Cell 004의 핵심 코드를 직접 작성하세요.
# 과목별 정답·해설지에는 원본 코드와 출제 의도가 있습니다.

pass


---
## 2) U-Net 모델 구현

- **목적:** U-Net의 표준 블록을 구현하고, Encoder/Decoder의 연결 구조를 코드로 확인합니다.
- **관찰 포인트**
  - `DoubleConv`(Conv–BN–ReLU 반복)이 feature 추출에 어떻게 쓰이는지
  - Down path에서 **Pooling/Stride**로 해상도가 줄어드는 지점
  - Up path에서 **Up-sampling(ConvTranspose2d 또는 bilinear)** 이 적용되는 지점
  - Skip 연결 시 **concat 채널 수**가 어떻게 증가하는지(가장 흔한 shape mismatch 원인)

> 구현을 따라가면서 각 단계의 feature map 크기(H×W)와 채널(C)을 메모해 두면 이해가 훨씬 빨라집니다.


In [ ]:
# ## 정답 입력
# Drill 2: 2) U-Net 모델 구현
# 원본 Cell 006의 핵심 코드를 직접 작성하세요.
# 과목별 정답·해설지에는 원본 코드와 출제 의도가 있습니다.

pass


---
## 3) 손실 함수 & 옵티마이저 설정

- **목적:** Segmentation 학습에 필요한 손실 함수와 최적화 방법을 설정합니다.
- **관찰 포인트**
  - Binary segmentation이면 보통 `BCEWithLogitsLoss`를 많이 사용(로짓 입력 주의)
  - Multi-class segmentation이면 `CrossEntropyLoss`(타깃은 class index) 사용
  - `sigmoid/softmax`를 **언제 적용해야 하는지**(loss와의 궁합)

> 이 템플릿은 “최소 구성” 예제를 우선 제공합니다.  
> 성능을 올리려면 Dice loss / IoU loss 같은 항목을 추가로 고려할 수 있습니다.


In [ ]:
# 손실 함수 및 옵티마이저 설정
criterion = nn.BCEWithLogitsLoss()  # Sigmoid + Binary Cross Entropy
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# (옵션) 학습 스케줄러
# 20 에포크(Epoch)마다 학습률을 변경, 기존 학습률에 0.5를 곱합
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=20, gamma=0.5)

---
## 4) 학습 루프

- **목적:** 가장 기본적인 형태의 학습/검증 루프를 실행해, loss가 정상적으로 내려가는지 확인합니다.
- **관찰 포인트**
  - `model.train()` / `model.eval()` 모드 전환
  - `optimizer.zero_grad()` → `loss.backward()` → `optimizer.step()` 순서
  - 검증 단계에서 `torch.no_grad()` 사용 여부
  - 학습이 진행되면서 train/val loss가 과도하게 벌어지는지(과적합 신호)

> Segmentation은 시각화가 쉬운 편이라, **숫자(loss) + 이미지(예측 결과)** 를 같이 보는 습관이 중요합니다.


In [ ]:
# ## 정답 입력
# Drill 3: 4) 학습 루프
# 원본 Cell 010의 핵심 코드를 직접 작성하세요.
# 과목별 정답·해설지에는 원본 코드와 출제 의도가 있습니다.

pass


---
## 4-1) 학습 곡선 시각화

- **목적:** epoch별 loss 변화를 그래프로 확인해 학습이 “정상적으로 진행되는지” 빠르게 점검합니다.
- **관찰 포인트**
  - 초반에 loss가 급격히 감소하는지
  - 어느 시점부터 감소가 둔화되는지(학습률 조정/early stopping 힌트)
  - train/val 곡선 간 격차가 커지는지(과적합)


In [ ]:
plt.figure()
plt.plot(train_losses, label='train_loss')
plt.plot(val_losses, label='val_loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.title('Training / Validation Loss')
plt.show()

---
## 5) 결과 시각화

- **목적:** 모델이 실제로 “분할”을 하고 있는지, 입력/정답/예측을 나란히 비교합니다.
- **관찰 포인트**
  - 출력이 로짓이면 `sigmoid` 후 threshold(예: 0.5)로 마스크를 만드는지
  - GT 마스크와 예측 마스크의 차이가 어디에서 주로 발생하는지(경계/작은 객체/노이즈 등)
  - 데이터가 간단한데도 실패한다면: 모델 출력 채널, loss, target 타입을 다시 점검

> 숫자(loss)만 보고 넘어가면 “뭔가 학습은 되는 것 같은데 결과가 이상한” 상황을 놓치기 쉽습니다.


In [ ]:
# ## 정답 입력
# Drill 4: 5) 결과 시각화
# 원본 Cell 014의 핵심 코드를 직접 작성하세요.
# 과목별 정답·해설지에는 원본 코드와 출제 의도가 있습니다.

pass
